# Milestone 6: Diffusion Product Image Generation

Goal: generate alternative product and lifestyle images from product metadata using diffusion models.

This notebook uses the processed Amazon Reviews splits, builds image-generation prompts from product title, description, and category fields, and saves generated images under `data/generated/`.

Milestone 7 is not implemented here.

## Setup

`FAST_MODE = True` keeps generation small. The notebook tries the preferred Stable Diffusion v1.5 model first and falls back to the tiny diffusers pipeline if loading fails. Set `TRY_TINY_MODEL_FIRST_IN_FAST_MODE = True` when you only want a very small smoke test.

In [ ]:
from __future__ import annotations

import ast
import json
import re
import textwrap
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

FAST_MODE = True
RANDOM_SEED = 42

PREFERRED_MODEL = "runwayml/stable-diffusion-v1-5"
FALLBACK_MODEL = "hf-internal-testing/tiny-stable-diffusion-pipe"
TRY_TINY_MODEL_FIRST_IN_FAST_MODE = False

if FAST_MODE:
    MAX_PRODUCTS_FOR_PROMPTS = 5
    MAX_PRODUCTS_TO_GENERATE = 1
    NUM_INFERENCE_STEPS = 6
    GUIDANCE_SCALE = 5.5
    IMAGE_HEIGHT = 384
    IMAGE_WIDTH = 384
else:
    MAX_PRODUCTS_FOR_PROMPTS = 30
    MAX_PRODUCTS_TO_GENERATE = 5
    NUM_INFERENCE_STEPS = 25
    GUIDANCE_SCALE = 7.5
    IMAGE_HEIGHT = 512
    IMAGE_WIDTH = 512

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
GENERATED_DIR = PROJECT_ROOT / "data" / "generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train.csv"
VALIDATION_PATH = PROCESSED_DIR / "validation.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Generated image directory: {GENERATED_DIR}")
print(f"FAST_MODE: {FAST_MODE}")

## Load Processed Splits

Load the processed train, validation, and test CSV files produced by Milestone 0. All three splits are combined only for prompt creation, not for modeling.

In [ ]:
for path in [TRAIN_PATH, VALIDATION_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required processed split: {path}")

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
validation_df = pd.read_csv(VALIDATION_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)

raw_products = pd.concat(
    [
        train_df.assign(split="train"),
        validation_df.assign(split="validation"),
        test_df.assign(split="test"),
    ],
    ignore_index=True,
)

print(f"Train rows: {len(train_df):,}")
print(f"Validation rows: {len(validation_df):,}")
print(f"Test rows: {len(test_df):,}")
print("Available columns:")
print(raw_products.columns.tolist())

## Detect Product Metadata Columns

The Amazon Reviews processed files can vary depending on preprocessing choices, so the notebook detects common title, description, category, and product ID fields safely.

In [ ]:
PRODUCT_ID_CANDIDATES = ["product_id", "parent_asin", "asin", "item_id"]
TITLE_CANDIDATES = ["product_title", "title", "name", "product_name"]
DESCRIPTION_CANDIDATES = ["product_description", "description", "details", "about_product"]
CATEGORY_CANDIDATES = ["main_category", "category", "categories", "product_category", "store"]


def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    return next((column for column in candidates if column in df.columns), None)


def clean_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def flatten_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    if isinstance(value, dict):
        return " ".join(flatten_text(item) for item in value.values())
    if isinstance(value, (list, tuple, set)):
        return " ".join(flatten_text(item) for item in value)
    text = str(value).strip()
    if text.startswith(("[", "{")):
        for parser in (json.loads, ast.literal_eval):
            try:
                return flatten_text(parser(text))
            except Exception:
                pass
    return clean_text(text)


product_id_col = first_existing_column(raw_products, PRODUCT_ID_CANDIDATES)
title_col = first_existing_column(raw_products, TITLE_CANDIDATES)
description_col = first_existing_column(raw_products, DESCRIPTION_CANDIDATES)
category_col = first_existing_column(raw_products, CATEGORY_CANDIDATES)

if title_col is None and description_col is None:
    raise KeyError(
        "No product title or description column found. "
        f"Available columns: {raw_products.columns.tolist()}"
    )

product_catalog = pd.DataFrame()
product_catalog["product_id"] = raw_products[product_id_col].astype(str) if product_id_col else raw_products.index.astype(str)
product_catalog["title"] = raw_products[title_col].map(flatten_text) if title_col else ""
product_catalog["description"] = raw_products[description_col].map(flatten_text) if description_col else ""
product_catalog["category"] = raw_products[category_col].map(flatten_text) if category_col else "All Beauty"

UNKNOWN_MARKERS = {"", "unknown", "nan", "none", "null", "[]", "{}"}
for metadata_column in ["title", "description", "category"]:
    product_catalog[metadata_column] = product_catalog[metadata_column].map(
        lambda value: "" if clean_text(value).lower() in UNKNOWN_MARKERS else clean_text(value)
    )

has_metadata = (
    product_catalog["title"].str.len().gt(0)
    | product_catalog["description"].str.len().gt(0)
    | product_catalog["category"].str.len().gt(0)
)
product_catalog = product_catalog[has_metadata].drop_duplicates("product_id").reset_index(drop=True)
product_catalog["title"] = product_catalog["title"].replace("", "All Beauty product")
product_catalog["category"] = product_catalog["category"].replace("", "All Beauty")
product_catalog = product_catalog.head(MAX_PRODUCTS_FOR_PROMPTS).copy()

print("Detected columns:")
print(
    {
        "product_id": product_id_col,
        "title": title_col,
        "description": description_col,
        "category": category_col,
    }
)
print(f"Prompt candidate products: {len(product_catalog):,}")
display(product_catalog.head())

## Create Diffusion Prompts

For each selected product, create three image prompts: a clean original product-photo prompt, a lifestyle variation, and a marketing/hero variation.

In [ ]:
def short_phrase(text: str, max_words: int = 18) -> str:
    words = clean_text(text).split()
    return " ".join(words[:max_words])


def safe_filename(text: str, max_chars: int = 80) -> str:
    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", text.lower()).strip("_")
    return safe[:max_chars] or "product"


def product_descriptor(row: pd.Series) -> str:
    title = short_phrase(row.get("title", ""), max_words=14)
    description = short_phrase(row.get("description", ""), max_words=18)
    category = short_phrase(row.get("category", "All Beauty"), max_words=8)
    parts = [title or "beauty product"]
    if description and description.lower() not in title.lower():
        parts.append(description)
    if category:
        parts.append(f"in the {category} category")
    return ", ".join(parts)


prompt_rows = []
selected_products = product_catalog.head(MAX_PRODUCTS_TO_GENERATE).copy()

for _, row in selected_products.iterrows():
    descriptor = product_descriptor(row)
    product_id = row["product_id"]
    prompt_rows.extend(
        [
            {
                "product_id": product_id,
                "title": row["title"],
                "variant": "original_prompt",
                "prompt": f"Professional product photography of {descriptor} on a clean white background, realistic packaging, soft studio lighting",
            },
            {
                "product_id": product_id,
                "title": row["title"],
                "variant": "lifestyle_variation",
                "prompt": f"Lifestyle photograph of {descriptor} in a calm modern bathroom, natural morning light, skincare routine, realistic editorial style",
            },
            {
                "product_id": product_id,
                "title": row["title"],
                "variant": "marketing_hero_variation",
                "prompt": f"Marketing hero image of {descriptor}, premium beauty brand campaign, elegant composition, clean background, high-end product photography",
            },
        ]
    )

prompt_df = pd.DataFrame(prompt_rows)
if prompt_df.empty:
    raise ValueError("No prompts were created. Check the processed product metadata columns.")

display(prompt_df[["product_id", "title", "variant", "prompt"]])

## Load Diffusion Pipeline

Preferred model: `runwayml/stable-diffusion-v1-5`.

Fallback model: `hf-internal-testing/tiny-stable-diffusion-pipe`.

If diffusers, torch, internet access, or model loading is unavailable, the notebook saves labeled placeholder images so later comparison and reflection cells still run. Placeholder output is not a substitute for diffusion output; it is only a failure-safe for local execution.

In [ ]:
pipe = None
torch = None
device = "cpu"
torch_dtype = None
loaded_model = "placeholder-fallback"
pipeline_log = []

if FAST_MODE and TRY_TINY_MODEL_FIRST_IN_FAST_MODE:
    model_candidates = [FALLBACK_MODEL, PREFERRED_MODEL]
else:
    model_candidates = [PREFERRED_MODEL, FALLBACK_MODEL]


def choose_device_and_dtype(torch_module):
    if torch_module.cuda.is_available():
        return "cuda", torch_module.float16
    mps_backend = getattr(torch_module.backends, "mps", None)
    if mps_backend is not None and mps_backend.is_available():
        return "mps", torch_module.float32
    return "cpu", torch_module.float32


try:
    import torch as torch_import
    from diffusers import StableDiffusionPipeline

    torch = torch_import
    device, torch_dtype = choose_device_and_dtype(torch)
    for model_id in model_candidates:
        started_at = time.perf_counter()
        try:
            candidate_pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch_dtype)
            candidate_pipe = candidate_pipe.to(device)
            if hasattr(candidate_pipe, "enable_attention_slicing"):
                candidate_pipe.enable_attention_slicing()
            if hasattr(candidate_pipe, "set_progress_bar_config"):
                candidate_pipe.set_progress_bar_config(disable=False)
            pipe = candidate_pipe
            loaded_model = model_id
            pipeline_log.append(
                {
                    "model_id": model_id,
                    "status": "loaded",
                    "device": device,
                    "seconds": round(time.perf_counter() - started_at, 2),
                    "message": "ready",
                }
            )
            break
        except Exception as model_error:
            pipeline_log.append(
                {
                    "model_id": model_id,
                    "status": "failed",
                    "device": device,
                    "seconds": round(time.perf_counter() - started_at, 2),
                    "message": str(model_error)[:300],
                }
            )
except Exception as import_error:
    pipeline_log.append(
        {
            "model_id": "diffusers/torch import",
            "status": "failed",
            "device": device,
            "seconds": 0,
            "message": str(import_error)[:300],
        }
    )

print(f"Loaded generation backend: {loaded_model}")
display(pd.DataFrame(pipeline_log))

## Generate and Save Images

Generate one image for each prompt variant and save it to `data/generated/`. The metadata table records the prompt, output path, backend, and generation time.

In [ ]:
NEGATIVE_PROMPT = "low quality, blurry, distorted text, watermark, extra limbs, deformed packaging"


def wrap_for_image(text: str, width: int = 34, max_lines: int = 12) -> str:
    lines = textwrap.wrap(clean_text(text), width=width)
    return "\n".join(lines[:max_lines])


def create_placeholder_image(prompt: str, output_path: Path, size: tuple[int, int]) -> Image.Image:
    image = Image.new("RGB", size, color=(245, 245, 242))
    draw = ImageDraw.Draw(image)
    try:
        font = ImageFont.truetype("arial.ttf", 16)
        small_font = ImageFont.truetype("arial.ttf", 12)
    except Exception:
        font = ImageFont.load_default()
        small_font = ImageFont.load_default()
    draw.rectangle((16, 16, size[0] - 16, size[1] - 16), outline=(120, 120, 120), width=2)
    draw.text((28, 28), "Diffusion unavailable", fill=(40, 40, 40), font=font)
    draw.text((28, 58), wrap_for_image(prompt, width=32, max_lines=16), fill=(70, 70, 70), font=small_font)
    image.save(output_path)
    return image


def generation_size() -> tuple[int, int]:
    if loaded_model == FALLBACK_MODEL:
        return 128, 128
    return IMAGE_WIDTH, IMAGE_HEIGHT


def make_generator(seed: int):
    if torch is None:
        return None
    try:
        generator_device = "cuda" if device == "cuda" else "cpu"
        return torch.Generator(device=generator_device).manual_seed(seed)
    except Exception:
        return None


def generate_image(prompt: str, output_path: Path, seed: int) -> dict:
    width, height = generation_size()
    started_at = time.perf_counter()
    status = "generated"
    error_message = ""
    try:
        if pipe is None:
            status = "placeholder"
            image = create_placeholder_image(prompt, output_path, (width, height))
        else:
            generator = make_generator(seed)
            result = pipe(
                prompt=prompt,
                negative_prompt=NEGATIVE_PROMPT,
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                height=height,
                width=width,
                generator=generator,
            )
            image = result.images[0]
            image.save(output_path)
    except Exception as generation_error:
        status = "placeholder_after_error"
        error_message = str(generation_error)[:300]
        image = create_placeholder_image(f"Generation failed: {error_message}. Prompt: {prompt}", output_path, (width, height))

    return {
        "image_path": str(output_path),
        "generation_time_seconds": round(time.perf_counter() - started_at, 2),
        "backend": loaded_model,
        "status": status,
        "error": error_message,
        "width": width,
        "height": height,
    }


generated_rows = []
for idx, row in prompt_df.iterrows():
    filename = f"{safe_filename(row['product_id'])}_{row['variant']}.png"
    output_path = GENERATED_DIR / filename
    result = generate_image(row["prompt"], output_path, seed=RANDOM_SEED + idx)
    generated_rows.append({**row.to_dict(), **result})

generated_df = pd.DataFrame(generated_rows)
metadata_path = GENERATED_DIR / "generation_metadata.csv"
generated_df.to_csv(metadata_path, index=False)

print(f"Saved generation metadata to: {metadata_path}")
display(generated_df[["product_id", "variant", "backend", "status", "generation_time_seconds", "image_path", "prompt", "error"]])

## Image Comparison Display

Compare the original prompt, lifestyle variation, and marketing/hero variation side by side.

In [ ]:
variant_order = ["original_prompt", "lifestyle_variation", "marketing_hero_variation"]
products_to_show = generated_df["product_id"].drop_duplicates().tolist()

fig, axes = plt.subplots(
    len(products_to_show),
    len(variant_order),
    figsize=(5 * len(variant_order), 4 * len(products_to_show)),
)
axes = np.array(axes).reshape(len(products_to_show), len(variant_order))

for row_idx, product_id in enumerate(products_to_show):
    for col_idx, variant in enumerate(variant_order):
        ax = axes[row_idx, col_idx]
        match = generated_df[(generated_df["product_id"] == product_id) & (generated_df["variant"] == variant)]
        ax.axis("off")
        ax.set_title(variant.replace("_", " "), fontsize=10)
        if match.empty:
            ax.text(0.5, 0.5, "missing", ha="center", va="center")
            continue
        image_path = Path(match.iloc[0]["image_path"])
        if image_path.exists():
            ax.imshow(Image.open(image_path))
        else:
            ax.text(0.5, 0.5, "file not found", ha="center", va="center")

plt.tight_layout()
plt.show()

## Reflection on Quality and Failure Modes

Diffusion-generated product images can be useful for ideation, but beauty products are high-risk for visual inaccuracies. Packaging text, ingredient claims, exact bottle shapes, and brand marks should not be trusted unless verified against real product assets.

In [ ]:
reflection_rows = [
    {
        "area": "Product fidelity",
        "observation": "Generated images may invent packaging, label text, ingredients, size, or container shape.",
        "mitigation": "Use generated images only as concept art unless validated against real catalog images.",
    },
    {
        "area": "Prompt sensitivity",
        "observation": "Small prompt changes can shift the scene from clean product photography to generic lifestyle imagery.",
        "mitigation": "Keep prompt templates controlled and log the exact prompt with each generated image.",
    },
    {
        "area": "Model/runtime limits",
        "observation": "CPU generation is slow, and fallback tiny models are useful for plumbing tests but not final image quality.",
        "mitigation": "Run the preferred Stable Diffusion model on GPU for real evaluation.",
    },
    {
        "area": "Responsible use",
        "observation": "Synthetic marketing images can imply product claims that are not supported by metadata or reviews.",
        "mitigation": "Review outputs manually and avoid using generated claims in customer-facing product pages.",
    },
]

reflection_df = pd.DataFrame(reflection_rows)
display(reflection_df)

## Stretch Goal: Augmentation Candidates for Milestone 2

Generated images could be considered as augmentation candidates for the Milestone 2 image classifier, but only with careful controls. They can increase visual diversity, yet they may also teach the vision model synthetic artifacts instead of real product signals.

In [ ]:
augmentation_discussion = pd.DataFrame(
    [
        {
            "candidate_use": "Class balancing",
            "possible_benefit": "Generate extra examples for underrepresented rating bands or product styles.",
            "risk": "Synthetic images may not match real Amazon image distribution.",
            "recommendation": "Use only in a controlled ablation study against a real-image-only baseline.",
        },
        {
            "candidate_use": "Robustness testing",
            "possible_benefit": "Test whether the Milestone 2 model reacts to background, lifestyle context, and lighting shifts.",
            "risk": "The model may learn prompt artifacts rather than product features.",
            "recommendation": "Use generated images for evaluation probes before using them for training.",
        },
        {
            "candidate_use": "Prototype visual search demos",
            "possible_benefit": "Provide visual placeholders when product images are missing.",
            "risk": "Placeholders can misrepresent the actual product.",
            "recommendation": "Label generated images clearly and never mix them with verified catalog photos.",
        },
    ]
)

display(augmentation_discussion)

## Completion

Milestone 6 is complete. Milestone 7 is not implemented.

In [ ]:
print("Milestone 6 Complete")